### Load file

In [1]:
import json
from pathlib import Path

project = Path(r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx")

attack_file = project / "data" / "raw" / "attack" / "enterprise-attack-19.1.json"

with open(attack_file, "r", encoding="utf-8") as f:
    attack_data = json.load(f)

### check structure

In [2]:
attack_data.keys()

dict_keys(['type', 'id', 'objects'])

### Extract objects

In [3]:
objects = attack_data["objects"]

len(objects)

25843

### Filter techniques

In [4]:
techniques = []

for obj in objects:

    if obj.get("type") == "attack-pattern":

        techniques.append(obj)

len(techniques)

858

### Inspect one technique

In [5]:
techniques[0]

{'type': 'attack-pattern',
 'spec_version': '2.1',
 'id': 'attack-pattern--0042a9f5-f053-4769-b3ef-9ad018dfa298',
 'created': '2020-01-14T17:18:32.126Z',
 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5',
 'revoked': False,
 'external_references': [{'source_name': 'mitre-attack',
   'url': 'https://attack.mitre.org/techniques/T1055/011',
   'external_id': 'T1055.011'},
  {'source_name': 'Elastic Process Injection July 2017',
   'description': 'Hosseini, A. (2017, July 18). Ten Process Injection Techniques: A Technical Survey Of Common And Trending Process Injection Techniques. Retrieved December 7, 2017.',
   'url': 'https://www.endgame.com/blog/technical-blog/ten-process-injection-techniques-technical-survey-common-and-trending-process'},
  {'source_name': 'MalwareTech Power Loader Aug 2013',
   'description': 'MalwareTech. (2013, August 13). PowerLoader Injection – Something truly amazing. Retrieved December 16, 2017.',
   'url': 'https://www.malwaretech.com/2013/08

### Extract T-codes

In [6]:
tech_map = {}

for obj in techniques:

    if "external_references" in obj:

        for ref in obj["external_references"]:

            if ref.get("source_name") == "mitre-attack":

                if "external_id" in ref:

                    tech_map[obj["id"]] = ref["external_id"]

len(tech_map)

858

-----------

## Extract CAPEC → ATT&CK mapping

### Find relationships

In [7]:
relationships = []

for obj in objects:

    if obj.get("type") == "relationship":

        relationships.append(obj)

len(relationships)

21025

### Inspect one relationship

In [8]:
relationships[0]

{'type': 'relationship',
 'spec_version': '2.1',
 'id': 'relationship--00038d0e-7fc7-41c3-9055-edb4d87ea912',
 'created': '2021-04-27T01:56:35.810Z',
 'created_by_ref': 'identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5',
 'revoked': False,
 'external_references': [{'source_name': 'CheckPoint Volatile Cedar March 2015',
   'description': 'Threat Intelligence and Research. (2015, March 30). VOLATILE CEDAR. Retrieved February 8, 2021.',
   'url': 'https://media.kasperskycontenthub.com/wp-content/uploads/sites/43/2015/03/20082004/volatile-cedar-technical-report.pdf'}],
 'object_marking_refs': ['marking-definition--fa42a846-8d90-4e51-bc29-71d5b4802168'],
 'modified': '2026-04-29T14:23:05.902Z',
 'description': " [Explosive](https://attack.mitre.org/software/S0569) has collected the MAC address from the victim's machine.(Citation: CheckPoint Volatile Cedar March 2015) ",
 'relationship_type': 'uses',
 'source_ref': 'malware--6a21e3a4-5ffe-4581-af9a-6a54c7536f44',
 'target_ref': 'attack-pattern

### Build CAPEC → ATT&CK edges

In [9]:
edges = []

### Extract mappings

In [10]:
for rel in relationships:

    if rel.get("relationship_type") == "related-to":

        source = rel.get("source_ref")
        target = rel.get("target_ref")

        # CAPEC IDs in ATT&CK usually appear in description links or mapping objects
        if "attack-pattern" in source and "attack-pattern" in target:

            # skip internal ATT&CK-to-ATT&CK links
            continue

        # we only keep CAPEC → ATT&CK style links later after validation
        edges.append([source, rel["relationship_type"], target])

### Extract ATT&CK lookup table

In [11]:
attack_lookup = {}

for obj in objects:

    if obj.get("type") == "attack-pattern":

        if "external_references" in obj:

            for ref in obj["external_references"]:

                if ref.get("source_name") == "mitre-attack":

                    attack_lookup[obj["id"]] = ref.get("external_id")

len(attack_lookup)

858

### Build CAPEC list

In [15]:
import xml.etree.ElementTree as ET
from pathlib import Path

project = Path(r"C:\Users\HP\Documents\python\bio informatics\final exam\ThreatGraphx")

capec_file = project / "data" / "raw" / "capec" / "capec_latest.xml"

tree = ET.parse(capec_file)
root = tree.getroot()

print("CAPEC loaded successfully")

CAPEC loaded successfully


In [16]:
capec_list = []

for elem in root.iter():

    if "Attack_Pattern" in elem.tag and "ID" in elem.attrib:

        capec_list.append({
            "capec_id": "CAPEC-" + elem.attrib["ID"],
            "name": elem.attrib.get("Name", "")
        })

len(capec_list)

615

In [17]:
edges = []

for capec in capec_list:

    capec_name = capec["name"].lower()

    for obj in objects:

        if obj.get("type") == "attack-pattern":

            if "name" in obj:

                atk_name = obj["name"].lower()

                # simple similarity check
                if any(word in atk_name for word in capec_name.split() if len(word) > 4):

                    if obj["id"] in attack_lookup:

                        edges.append([
                            capec["capec_id"],
                            "maps_to",
                            attack_lookup[obj["id"]]
                        ])

In [18]:
len(edges)

5845

### Convert to DataFrame

In [19]:
import pandas as pd

capec_attack_df = pd.DataFrame(edges, columns=["source", "relation", "target"])

capec_attack_df.head()

,source,relation,target
0,CAPEC-10,maps_to,T1574.007
1,CAPEC-10,maps_to,T1480.001
2,CAPEC-101,maps_to,T1055.011
3,CAPEC-101,maps_to,T1583.007
4,CAPEC-101,maps_to,T1583.002


### Save

In [20]:
output_file = project / "data" / "processed" / "capec_attack_edges.csv"

capec_attack_df.to_csv(output_file, index=False)

print("saved")

saved
